In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/MolTrans/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/MolTrans'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/MolTrans/script')



In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils import data
from sklearn.metrics import roc_auc_score, average_precision_score
import codecs
from subword_nmt.apply_bpe import BPE

from models import BIN_Interaction_Flat
from config import BIN_config_DBPE

# ==================== Data preprocessing ====================

vocab_path_protein = _kp('external/MolTrans/ESPF/protein_codes_uniprot.txt')
bpe_codes_protein = codecs.open(vocab_path_protein)
pbpe = BPE(bpe_codes_protein, merges=-1, separator='')
sub_csv_p = pd.read_csv(_kp('external/MolTrans/ESPF/subword_units_map_uniprot.csv'))
idx2word_p = sub_csv_p['index'].values
words2idx_p = dict(zip(idx2word_p, range(0, len(idx2word_p))))

vocab_path_drug = _kp('external/MolTrans/ESPF/drug_codes_chembl.txt')
bpe_codes_drug = codecs.open(vocab_path_drug)
dbpe = BPE(bpe_codes_drug, merges=-1, separator='')
sub_csv_d = pd.read_csv(_kp('external/MolTrans/ESPF/subword_units_map_chembl.csv'))
idx2word_d = sub_csv_d['index'].values
words2idx_d = dict(zip(idx2word_d, range(0, len(idx2word_d))))


def protein2emb_encoder(x):
    max_p = 545
    t1 = pbpe.process_line(x).split()
    try:
        i1 = np.asarray([words2idx_p[i] for i in t1])
    except:
        i1 = np.array([0])
    l = len(i1)
    if l < max_p:
        i = np.pad(i1, (0, max_p - l), 'constant', constant_values=0)
        input_mask = ([1] * l) + ([0] * (max_p - l))
    else:
        i = i1[:max_p]
        input_mask = [1] * max_p
    return i, np.asarray(input_mask)


def drug2emb_encoder(x):
    max_d = 50
    t1 = dbpe.process_line(x).split()
    try:
        i1 = np.asarray([words2idx_d[i] for i in t1])
    except:
        i1 = np.array([0])
    l = len(i1)
    if l < max_d:
        i = np.pad(i1, (0, max_d - l), 'constant', constant_values=0)
        input_mask = ([1] * l) + ([0] * (max_d - l))
    else:
        i = i1[:max_d]
        input_mask = [1] * max_d
    return i, np.asarray(input_mask)


class BIN_Data_Encoder_Predict(data.Dataset):
    def __init__(self, list_IDs, labels, df_dti):
        self.labels = labels
        self.list_IDs = list_IDs
        self.df = df_dti

    def __len__(self):
        return len(self.list_IDs)

    def __getitem__(self, index):
        index = self.list_IDs[index]
        d = self.df.iloc[index]['Ligand']
        p = self.df.iloc[index]['Protein']
        d_v, input_mask_d = drug2emb_encoder(d)
        p_v, input_mask_p = protein2emb_encoder(p)
        y = self.labels[index]
        return d_v, p_v, input_mask_d, input_mask_p, y


# ==================== Metric calculation ====================

# def calculate_bedroc(y_true, y_pred, alpha=20.0):
#     sorted_indices = np.argsort(y_pred)[::-1]
#     sorted_labels = y_true[sorted_indices]
#     n = len(y_true)
#     n_actives = np.sum(y_true)
#     if n_actives == 0 or n_actives == n:
#         return np.nan
#     ra = n_actives / n
#     sum_weighted_actives = sum(
#         np.exp(-alpha * (i + 1) / n)
#         for i, label in enumerate(sorted_labels) if label == 1
#     )
#     sum_random = ra * (1 - np.exp(-alpha)) / (np.exp(alpha / n) - 1)
#     sum_ideal = (1 - np.exp(-alpha * ra)) / (1 - np.exp(-alpha / n))
#     if sum_ideal == sum_random:
#         return np.nan
#     return (sum_weighted_actives - sum_random) / (sum_ideal - sum_random)

# import numpy as np
# from rdkit.ML.Scoring.Scoring import CalcBEDROC

# def calculate_bedroc(y_true, y_pred, alpha=80.5):
#     y_true = np.asarray(y_true).astype(int)
#     y_pred = np.asarray(y_pred).astype(float)

#     # Enforce binary 0/1 labels
#     if not np.all((y_true == 0) | (y_true == 1)):
#         raise ValueError("y_true must be binary 0/1 for BEDROC.")

#     n_actives = int(y_true.sum())
#     if n_actives == 0 or n_actives == len(y_true):
#         return np.nan

#     # Build (score, label) pairs and sort by descending score
#     data = list(zip(y_pred.tolist(), y_true.tolist()))
#     data.sort(key=lambda x: x[0], reverse=True)

#     # col=1 selects the label in the second column
#     return CalcBEDROC(data, col=1, alpha=alpha)

def calculate_bedroc(y_true, y_score, bedroc_alpha):
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0                    # Return 0 when there are no active compounds

    # 1. Sort predictions and obtain the ranks of active compounds
    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]      # 1-based
    x      = (ranks - 0.5) / N                          # Rank midpoint

    # 2. Calculate RIE
    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    # 3. Calculate the limits
    x_max  = (np.arange(1, n_act+1) - 0.5) / N          # Best case
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N      # Worst case
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    # 4. Normalize
    return (rie - rie_min) / (rie_max - rie_min)


def calculate_ef(y_true, y_pred, percentage):
    """Calculate EF at a single percentage."""
    sorted_indices = np.argsort(y_pred)[::-1]
    sorted_labels = y_true[sorted_indices]
    total = len(y_true)
    total_actives = np.sum(y_true)
    if total_actives == 0:
        return np.nan
    n_top = max(1, int(np.ceil(total * percentage / 100.0)))
    actives_in_top = np.sum(sorted_labels[:n_top])
    expected = n_top * (total_actives / total)
    if expected == 0:
        return np.nan
    return actives_in_top / expected


def calculate_target_metrics(y_true, y_pred, ef_percentages):
    """Calculate all metrics for one target with one model."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    n = len(y_true)
    n_actives = int(np.sum(y_true))

    metrics = {}

    # ROC_AUC requires both positive and negative samples
    if n_actives > 0 and n_actives < n:
        metrics['ROC_AUC'] = roc_auc_score(y_true, y_pred)
        metrics['PR_AUC'] = average_precision_score(y_true, y_pred)
        metrics['BEDROC'] = calculate_bedroc(y_true, y_pred, bedroc_alpha=80.5)
    else:
        metrics['ROC_AUC'] = np.nan
        metrics['PR_AUC'] = np.nan
        metrics['BEDROC'] = np.nan

    for pct in ef_percentages:
        key = f'EF_{pct}%'
        if n_actives > 0:
            metrics[key] = calculate_ef(y_true, y_pred, pct)
        else:
            metrics[key] = np.nan

    return metrics


# ==================== Model loading and prediction ====================

def load_model(model_path, device):
    config = BIN_config_DBPE()
    model = BIN_Interaction_Flat(**config)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model, config


def predict(model, data_loader, config_batch_size, device):
    """Return prediction probabilities in data_loader order."""
    y_pred = []
    model.eval()
    with torch.no_grad():
        for d, p, d_mask, p_mask, label in data_loader:
            bs = d.size(0)
            if bs < config_batch_size:
                pad = config_batch_size - bs
                d      = torch.cat([d,      d[:pad]],      dim=0)
                p      = torch.cat([p,      p[:pad]],      dim=0)
                d_mask = torch.cat([d_mask, d_mask[:pad]], dim=0)
                p_mask = torch.cat([p_mask, p_mask[:pad]], dim=0)

            score = model(
                d.long().to(device),
                p.long().to(device),
                d_mask.long().to(device),
                p_mask.long().to(device)
            )
            logits = torch.squeeze(torch.sigmoid(score))
            if logits.dim() == 0:
                logits = logits.unsqueeze(0)
            y_pred.extend(logits[:bs].detach().cpu().numpy().flatten().tolist())

    return np.array(y_pred)


# ==================== Main evaluation function ====================

def prepare_test_df(test_data_path, kiba_threshold):
    """Load and standardize test columns and generate classification_label."""
    df = pd.read_csv(test_data_path)

    col_map_ligand = {'SMILES': 'Ligand', 'compound_iso_smiles': 'Ligand', 'smiles': 'Ligand'}
    col_map_protein = {'target_sequence': 'Protein', 'protein': 'Protein'}
    col_map_kiba   = {'kiba': 'KIBA'}

    for old, new in {**col_map_ligand, **col_map_protein, **col_map_kiba}.items():
        if old in df.columns and new not in df.columns:
            df.rename(columns={old: new}, inplace=True)

    if 'KIBA' not in df.columns:
        raise ValueError("Cannot find KIBA column in test data")
    if 'uniprot_id' not in df.columns:
        raise ValueError("Cannot find uniprot_id column in test data")

    df['classification_label'] = (df['KIBA'] <= kiba_threshold).astype(int)
    df = df.reset_index(drop=True)
    return df


def evaluate_per_target(
    model_dir,
    model_pattern,
    model_ids,
    test_data_path,
    output_path,
    batch_size=16,
    device='cuda:0',
    kiba_threshold=3.0,
    ef_percentages=[0.5, 1, 2, 5, 10]
):
    """
    Calculate metric means and standard deviations across five models for each target.
    Return the results in one table.
    """
    if not torch.cuda.is_available() and device.startswith('cuda'):
        print("⚠️  CUDA not available, falling back to CPU")
        device = 'cpu'

    print("="*80)
    print("🚀 Per-Target Evaluation Across Models")
    print(f"   Device        : {device}")
    print(f"   KIBA threshold: ≤ {kiba_threshold}")
    print(f"   EF percentages: {ef_percentages}%")
    print(f"   Models        : {model_ids}")
    print("="*80)

    # Prepare the test data
    df_test = prepare_test_df(test_data_path, kiba_threshold)
    targets = df_test['uniprot_id'].unique()
    print(f"\n📊 Test set: {len(df_test)} samples | {len(targets)} unique targets\n")

    # Build the DataLoader once and reuse it across models
    dataset = BIN_Data_Encoder_Predict(
        df_test.index.values,
        df_test['classification_label'].values,
        df_test
    )
    loader = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False
    )

    # Collect predictions from each model for the complete test set
    # model_preds[model_id] = np.array of shape (N,)
    model_preds = {}

    for model_id in model_ids:
        model_filename = model_pattern.format(model_id)
        model_path = os.path.join(model_dir, model_filename)

        if not os.path.exists(model_path):
            print(f"⚠️  Model not found, skipping: {model_path}")
            continue

        print(f"📦 Running model {model_id}: {model_filename}")
        model, config = load_model(model_path, device)
        preds = predict(model, loader, config['batch_size'], device)
        model_preds[model_id] = preds
        print(f"   ✅ Done ({len(preds)} predictions)")

    valid_model_ids = list(model_preds.keys())
    print(f"\n✅ Successfully loaded {len(valid_model_ids)} / {len(model_ids)} models\n")

    if len(valid_model_ids) == 0:
        print("❌ No models were successfully loaded!")
        return None

    labels = df_test['classification_label'].values  # shape (N,)

    # Calculate per-target metrics
    ef_keys = [f'EF_{p}%' for p in ef_percentages]
    base_keys = ['ROC_AUC', 'PR_AUC', 'BEDROC'] + ef_keys

    rows = []

    for target_id in sorted(targets):
        mask = (df_test['uniprot_id'] == target_id).values
        idx  = np.where(mask)[0]

        y_true_t = labels[idx]
        n        = len(y_true_t)
        n_act    = int(np.sum(y_true_t))
        act_ratio = n_act / n if n > 0 else np.nan

        # Calculate metrics for this target with each model
        per_model_metrics = {k: [] for k in base_keys}

        for mid in valid_model_ids:
            y_pred_t = model_preds[mid][idx]
            m = calculate_target_metrics(y_true_t, y_pred_t, ef_percentages)
            for k in base_keys:
                per_model_metrics[k].append(m[k])

        # Aggregate means and standard deviations, ignoring NaN values
        row = {
            'Target'      : target_id,
            'N'           : n,
            'Actives'     : n_act,
            'ActiveRatio' : round(act_ratio, 4),
        }
        for k in base_keys:
            vals = [v for v in per_model_metrics[k] if not np.isnan(v)]
            row[f'{k}_mean'] = round(np.mean(vals), 4) if vals else np.nan
            row[f'{k}_std']  = round(np.std(vals,  ddof=1), 4) if len(vals) > 1 else np.nan

        rows.append(row)

    # Output column order
    fixed_cols = ['Target', 'N', 'Actives', 'ActiveRatio']
    metric_cols = []
    for k in base_keys:
        metric_cols += [f'{k}_mean', f'{k}_std']

    df_out = pd.DataFrame(rows, columns=fixed_cols + metric_cols)

    # Print the aggregate results
    print("="*80)
    print("📋 Per-Target Results")
    print("="*80)
    print(df_out.to_string(index=False))

    # Append a row containing the global means
    summary_row = {'Target': 'ALL_TARGETS_MEAN', 'N': df_out['N'].sum(),
                   'Actives': df_out['Actives'].sum(),
                   'ActiveRatio': round(df_out['ActiveRatio'].mean(), 4)}
    for col in metric_cols:
        summary_row[col] = round(df_out[col].mean(skipna=True), 4)

    df_summary_row = pd.DataFrame([summary_row], columns=fixed_cols + metric_cols)
    df_out = pd.concat([df_out, df_summary_row], ignore_index=True)

    # Save
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_out.to_csv(output_path, index=False)
    print(f"\n💾 Results saved to: {output_path}")
    print("\n✅ Done!")

    return df_out


# ==================== Entry point ====================

if __name__ == '__main__':

    model_dir   = _kp('model/MolTrans')
    model_pattern = '10_50_{}_best_model.pt'
    model_ids   = [1, 2, 3, 4, 5]

    test_data_path = _kp('dataset/KIBA_seen_LUDe_augment_sim08.csv')

    output_path = _kp('others/baseline/MolTrans/results/KIBA_seen_LUDe_augment/KIBA_seen_LUDe_augment_sim08_result.csv')

    df_out = evaluate_per_target(
        model_dir     = model_dir,
        model_pattern = model_pattern,
        model_ids     = model_ids,
        test_data_path= test_data_path,
        output_path   = output_path,
        batch_size    = 16,
        device        = 'cuda:0',
        kiba_threshold= 3.0,
        ef_percentages= [0.5, 1, 2, 5, 10],
    )

🚀 Per-Target Evaluation Across Models
   Device        : cuda:0
   KIBA threshold: ≤ 3.0
   EF percentages: [0.5, 1, 2, 5, 10]%
   Models        : [1, 2, 3, 4, 5]


/tmp/ipykernel_4178745/2311113291.py:242: DtypeWarning: Columns (0,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(test_data_path)



📊 Test set: 1044260 samples | 172 unique targets

📦 Running model 1: 10_50_1_best_model.pt


/tmp/ipykernel_4178745/2311113291.py:203: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)


   ✅ Done (1044260 predictions)
📦 Running model 2: 10_50_2_best_model.pt
   ✅ Done (1044260 predictions)
📦 Running model 3: 10_50_3_best_model.pt
   ✅ Done (1044260 predictions)
📦 Running model 4: 10_50_4_best_model.pt
   ✅ Done (1044260 predictions)
📦 Running model 5: 10_50_5_best_model.pt
   ✅ Done (1044260 predictions)

✅ Successfully loaded 5 / 5 models

📋 Per-Target Results
Target     N  Actives  ActiveRatio  ROC_AUC_mean  ROC_AUC_std  PR_AUC_mean  PR_AUC_std  BEDROC_mean  BEDROC_std  EF_0.5%_mean  EF_0.5%_std  EF_1%_mean  EF_1%_std  EF_2%_mean  EF_2%_std  EF_5%_mean  EF_5%_std  EF_10%_mean  EF_10%_std
O00444  3610      160       0.0443        0.6828       0.0200       0.0732      0.0029       0.0498      0.0195        0.9500       0.9935      0.9757     0.6953      1.1127     0.4147      1.1718     0.4282       1.2375      0.1734
O14757 23209     1180       0.0508        0.6900       0.0391       0.0786      0.0094       0.0216      0.0093        0.2017       0.1406      0.3208  